In [3]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

import time

project_root = Path(
    r"C:\Users\rajur\OneDrive\Desktop\next-move-ai"
)

feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_4_question_difficulty"
    / "base_features"
)

prediction_dir = (
    project_root
    / "data"
    / "processed"
    / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Project root: C:\Users\rajur\OneDrive\Desktop\next-move-ai
Feature directory: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_4_question_difficulty\base_features
Prediction directory: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [4]:
fold_number = 1

train_df = pd.read_csv(
    feature_dir / f"fold_{fold_number}_train.csv"
)

val_df = pd.read_csv(
    feature_dir / f"fold_{fold_number}_validation.csv"
)

numeric_features = [
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

categorical_features = [
    "answer_type"
]

target_column = "correct"

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTraining answer types:")
print(train_df["answer_type"].value_counts(dropna=False))

print("\nValidation answer types:")
print(val_df["answer_type"].value_counts(dropna=False))

print("\nMissing answer types:")
print("Training:", train_df["answer_type"].isna().sum())
print("Validation:", val_df["answer_type"].isna().sum())

Training shape: (177324, 13)
Validation shape: (42659, 13)

Training answer types:
answer_type
algebra          146002
fill_in_1         17826
choose_1          13427
choose_n             65
open_response         4
Name: count, dtype: int64

Validation answer types:
answer_type
algebra          35505
fill_in_1         3754
choose_1          3396
open_response        4
Name: count, dtype: int64

Missing answer types:
Training: 0
Validation: 0


In [5]:
numeric_features = [
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

categorical_features = [
    "answer_type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42
    ))
])

print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['question_smoothed_accuracy',
                                                   'log_question_previous_interactions']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['answer_type'])])),
                ('logistic_regression',
                 LogisticRegression(max_iter=1000, random_state=42))])


In [6]:

fold_results = []
all_predictions = []

numeric_features = [
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

categorical_features = ["answer_type"]

for fold_number in range(1, 6):

    print(f"\n{'='*45}")
    print(f"Training V4.3 — Fold {fold_number}")
    print(f"{'='*45}")

    # Load fold-specific data
    train_df = pd.read_csv(
        feature_dir / f"fold_{fold_number}_train.csv"
    )

    val_df = pd.read_csv(
        feature_dir / f"fold_{fold_number}_validation.csv"
    )

    X_train = train_df[
        numeric_features + categorical_features
    ]

    y_train = train_df["correct"]

    X_val = val_df[
        numeric_features + categorical_features
    ]

    y_val = val_df["correct"]

    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_features
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_features
            )
        ]
    )

    # Complete pipeline
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train
    start_time = time.perf_counter()

    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start_time

    # Predict probabilities
    start_time = time.perf_counter()

    y_pred_proba = model.predict_proba(X_val)[:, 1]

    prediction_time = time.perf_counter() - start_time

    # Evaluate
    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier,
        "training_time": training_time,
        "prediction_time": prediction_time
    })

    # Store OOF predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].to_numpy(),
        "order_id": val_df["order_id"].to_numpy(),
        "fold": fold_number,
        "y_true": y_val.to_numpy(),
        "y_pred_proba": y_pred_proba,
        "model": "v4.3"
    })

    all_predictions.append(fold_predictions)

    print(f"Training interactions: {len(train_df)}")
    print(f"Validation interactions: {len(val_df)}")
    print(f"Log Loss: {fold_log_loss:.6f}")
    print(f"Brier Score: {fold_brier:.6f}")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Prediction time: {prediction_time:.2f} seconds")



Training V4.3 — Fold 1
Training interactions: 177324
Validation interactions: 42659
Log Loss: 0.617874
Brier Score: 0.214086
Training time: 0.17 seconds
Prediction time: 0.01 seconds

Training V4.3 — Fold 2
Training interactions: 176273
Validation interactions: 43710
Log Loss: 0.607684
Brier Score: 0.209215
Training time: 0.14 seconds
Prediction time: 0.01 seconds

Training V4.3 — Fold 3
Training interactions: 170186
Validation interactions: 49797
Log Loss: 0.625957
Brier Score: 0.217801
Training time: 0.13 seconds
Prediction time: 0.01 seconds

Training V4.3 — Fold 4
Training interactions: 178311
Validation interactions: 41672
Log Loss: 0.613507
Brier Score: 0.212061
Training time: 0.15 seconds
Prediction time: 0.01 seconds

Training V4.3 — Fold 5
Training interactions: 177838
Validation interactions: 42145
Log Loss: 0.608146
Brier Score: 0.209559
Training time: 0.13 seconds
Prediction time: 0.01 seconds


In [7]:
fold_results_df = pd.DataFrame(fold_results)

print("FOLD-WISE RESULTS")
print(fold_results_df.round(6).to_string(index=False))

print("\nAVERAGE CROSS-VALIDATION RESULTS")

print("Average Log Loss:", fold_results_df["log_loss"].mean())

print("Average Brier Score:", fold_results_df["brier_score"].mean())

FOLD-WISE RESULTS
 fold  log_loss  brier_score  training_time  prediction_time
    1  0.617874     0.214086       0.172354         0.010436
    2  0.607684     0.209215       0.137874         0.012420
    3  0.625957     0.217801       0.133188         0.012015
    4  0.613507     0.212061       0.145260         0.011156
    5  0.608146     0.209559       0.134246         0.010699

AVERAGE CROSS-VALIDATION RESULTS
Average Log Loss: 0.6146336428115794
Average Brier Score: 0.21254426256479123


In [8]:
oof_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

prediction_path = prediction_dir / "v4_3_predictions.csv"

oof_predictions.to_csv(
    prediction_path,
    index=False
)

print("OOF prediction shape:", oof_predictions.shape)
print("Saved to:", prediction_path)
print("Missing values:", oof_predictions.isna().sum().sum())

print(
    "Duplicate interactions:",
    oof_predictions.duplicated(
        ["user_id", "order_id"]
    ).sum()
)

OOF prediction shape: (219983, 6)
Saved to: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v4_3_predictions.csv
Missing values: 0
Duplicate interactions: 0


In [9]:
import pandas as pd

# Load the saved V4.3 out-of-fold predictions
predictions = pd.read_csv(
    "../../../data/processed/model_predictions/v4_3_predictions.csv"
)

# Basic information
print("Shape:", predictions.shape)
print("\nColumns:")
print(predictions.columns.tolist())

# Check missing values
print("\nMissing values:")
print(predictions.isnull().sum())

# Check duplicate interactions
print(
    "\nDuplicate (user_id, order_id) pairs:",
    predictions.duplicated(subset=["user_id", "order_id"]).sum()
)

# Check number of unique students and interactions
print("\nUnique students:", predictions["user_id"].nunique())
print("Unique interactions:", predictions[["user_id", "order_id"]].drop_duplicates().shape[0])

Shape: (219983, 6)

Columns:
['user_id', 'order_id', 'fold', 'y_true', 'y_pred_proba', 'model']

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Duplicate (user_id, order_id) pairs: 0

Unique students: 3373
Unique interactions: 219983
